# 02 Random Forest: Predict Penguin Species

Random Forest can learn non-linear relationships and report feature importance. Here we compare it with Logistic Regression.


In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE = 0.25

from sklearn.ensemble import RandomForestClassifier


## 1. Load Data


In [ ]:
candidate_paths = [
    Path('../../data/ml_data/penguins.csv'),
    Path('../data/ml_data/penguins.csv'),
    Path('/workspace/data/ml_data/penguins.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/penguins.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find penguins.csv. Please check data/ml_data/penguins.csv')

penguins = pd.read_csv(data_path)
print('Data path:', data_path)
print('Data shape:', penguins.shape)
penguins.head()


## 2. Prepare Training and Test Sets


In [ ]:
target_col = 'species'
drop_cols = ['Unnamed: 0']
feature_cols = [col for col in penguins.columns if col not in drop_cols + [target_col]]

X = penguins[feature_cols].copy()
y = penguins[target_col].copy()

numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'year']
categorical_features = ['island', 'sex']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Test-set class distribution:')
print(y_test.value_counts().sort_index())


## 3. Build the Preprocessing and Random Forest Pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
])

random_forest.fit(X_train, y_train)
print(random_forest)


## 4. Predict and Evaluate


In [ ]:
y_pred = random_forest.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Random Forest Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.show()

pd.DataFrame({'actual': y_test.values, 'predicted': y_pred}, index=y_test.index).head(12)


## 5. Feature Importance

Random Forest feature importance is not causal explanation. It only shows which features the trees used more often for splits.


In [ ]:
encoded_feature_names = random_forest.named_steps['preprocessor'].get_feature_names_out()
importance = pd.Series(
    random_forest.named_steps['model'].feature_importances_,
    index=encoded_feature_names,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index, palette='Greens_r', legend=False, ax=ax)
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.show()

importance.to_frame('importance').round(4)


## 6. Summary

Random Forest often captures non-linear combinations among measurement features. If it is close to Logistic Regression, the main class differences are already well described by mostly linear boundaries.
